In [ ]:
# refer : https://www.kaggle.com/code/favorcat/nsmc-day4-4
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# 한글폰트 설정

In [ ]:
from matplotlib import font_manager
plt.rcParams['axes.unicode_minus'] = False
fontpath = '/kaggle/input/nanumfont/NanumGothic.ttf'
fontprop = font_manager.FontProperties(fname=fontpath,size=12)

# nsmc read

In [ ]:
train_data = pd.read_csv(
    "/kaggle/input/naive-bayse-sentiment-analysis-naver-movie/ratings_train.txt"
    , header=0
    , delimiter="\t"
    , quoting=3)

In [ ]:
train_data.info

In [ ]:
train_length=train_data['document'].astype(str).apply(len)

In [ ]:
train_length.describe()

In [ ]:
plt.figure(figsize=(12,5))
plt.hist(train_length, bins=200, color="red", label='document')

In [ ]:
plt.figure(figsize=(12,5))
plt.hist(train_length, bins=200, color="red", label='document')
plt.yscale('log')

In [ ]:
plt.figure(figsize=(12,5))
plt.boxplot(train_length, labels=['counts'], showmeans=True)

In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud
cloud = WordCloud(font_path=fontpath, width=800, height=600).generate(" ".join(train_data['document'].astype(str)))
plt.figure(figsize=(20,15))
plt.imshow(cloud)
plt.axis('off')

In [ ]:
from wordcloud import WordCloud
cloud = WordCloud(font_path=fontpath, width=800, height=600).generate(" ".join(train_data['document'].astype(str).loc[train_data['label']==0]))
plt.figure(figsize=(20,15))
plt.imshow(cloud)
plt.axis('off')

In [ ]:
from wordcloud import WordCloud
cloud = WordCloud(font_path=fontpath, width=800, height=600).generate(" ".join(train_data['document'].astype(str).loc[train_data['label']==1]))
plt.figure(figsize=(20,15))
plt.imshow(cloud)
plt.axis('off')

# 전처리

# hanja

In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install hanja

In [ ]:
import hanja
import re

In [ ]:
import hanja
import re

# http link 제거

In [ ]:
import re
pattern_http = re.compile(r'http://[a-zA-Z0-9.?/&=:]*')

In [ ]:
links = np.mean(train_data['document'].astype(str).apply(lambda x: pattern_http.match(x) != None ))
print(f'링크가 있는 문장: {round(links*100,2)}%')

In [ ]:
train_data['document'].astype(str).loc[train_data['document'].astype(str).apply(lambda x: pattern_http.match(x) != None )]

In [ ]:
clean_link = re.sub(r'http://[a-zA-Z0-9.?/&=:]*', "",  'http://blog.naver.com/dicasso6 이영화를 좋아하시는 분들 초대')
clean_link.strip()

# 특수문자 제거

In [ ]:
review_data2 = train_data['document'][19233]
review_data2

In [ ]:
 re.sub("[^가-힣ㄱ-ㅎㅏ-ㅣ0-9a-zA-Z]", " ", review_data2).strip()

# 불용어

In [ ]:
stop_words = """
아
휴
아이구
아이쿠
아이고
어
나
우리
저희
따라
의해
을
를
에
의
가
으로
로
에게
뿐이다
의거하여
근거하여
입각하여
기준으로
예하면
예를 들면
예를 들자면
저
소인
소생
저희
지말고
하지마
하지마라
다른
물론
또한
그리고
비길수 없다
해서는 안된다
뿐만 아니라
만이 아니다
만은 아니다
막론하고
관계없이
그치지 않다
그러나
그런데
하지만
든간에
논하지 않다
따지지 않다
설사
비록
더라도
아니면
만 못하다
하는 편이 낫다
불문하고
향하여
향해서
향하다
쪽으로
틈타
이용하여
타다
오르다
제외하고
이 외에
이 밖에
하여야
비로소
한다면 몰라도
외에도
이곳
여기
부터
기점으로
따라서
할 생각이다
하려고하다
이리하여
그리하여
그렇게 함으로써
하지만
일때
할때
앞에서
중에서
보는데서
으로써
로써
까지
해야한다
일것이다
반드시
할줄알다
할수있다
할수있어
임에 틀림없다
한다면
등
등등
제
겨우
단지
다만
할뿐
딩동
댕그
대해서
대하여
대하면
훨씬
얼마나
얼마만큼
얼마큼
남짓
여
얼마간
약간
다소
좀
조금
다수
몇
얼마
지만
하물며
또한
그러나
그렇지만
하지만
이외에도
대해 말하자면
뿐이다
다음에
반대로
반대로 말하자면
이와 반대로
바꾸어서 말하면
바꾸어서 한다면
만약
그렇지않으면
까악
툭
딱
삐걱거리다
보드득
비걱거리다
꽈당
응당
해야한다
에 가서
각
각각
여러분
각종
각자
제각기
하도록하다
와
과
그러므로
그래서
고로
한 까닭에
하기 때문에
거니와
이지만
대하여
관하여
관한
과연
실로
아니나다를가
생각한대로
진짜로
한적이있다
하곤하였다
하
하하
허허
아하
거바
와
오
왜
어째서
무엇때문에
어찌
하겠는가
무슨
어디
어느곳
더군다나
하물며
더욱이는
어느때
언제
야
이봐
어이
여보시오
흐흐
흥
휴
헉헉
헐떡헐떡
영차
여차
어기여차
끙끙
아야
앗
아야
콸콸
졸졸
좍좍
뚝뚝
주룩주룩
솨
우르르
그래도
또
그리고
바꾸어말하면
바꾸어말하자면
혹은
혹시
답다
및
그에 따르는
때가 되어
즉
지든지
설령
가령
하더라도
할지라도
일지라도
지든지
몇
거의
하마터면
인젠
이젠
된바에야
된이상
만큼	어찌됏든
그위에
게다가
점에서 보아
비추어 보아
고려하면
하게될것이다
일것이다
비교적
좀
보다더
비하면
시키다
하게하다
할만하다
의해서
연이서
이어서
잇따라
뒤따라
뒤이어
결국
의지하여
기대여
통하여
자마자
더욱더
불구하고
얼마든지
마음대로
주저하지 않고
곧
즉시
바로
당장
하자마자
밖에 안된다
하면된다
그래
그렇지
요컨대
다시 말하자면
바꿔 말하면
즉
구체적으로
말하자면
시작하여
시초에
이상
허
헉
허걱
바와같이
해도좋다
해도된다
게다가
더구나
하물며
와르르
팍
퍽
펄렁
동안
이래
하고있었다
이었다
에서
로부터
까지
예하면
했어요
해요
함께
같이
더불어
마저
마저도
양자
모두
습니다
가까스로
하려고하다
즈음하여
다른
다른 방면으로
해봐요
습니까
했어요
말할것도 없고
무릎쓰고
개의치않고
하는것만 못하다
하는것이 낫다
매
매번
들
모
어느것
어느
로써
갖고말하자면
어디
어느쪽
어느것
어느해
어느 년도
라 해도
언젠가
어떤것
어느것
저기
저쪽
저것
그때
그럼
그러면
요만한걸
그래
그때
저것만큼
그저
이르기까지
할 줄 안다
할 힘이 있다
너
너희
당신
어찌
설마
차라리
할지언정
할지라도
할망정
할지언정
구토하다
게우다
토하다
메쓰겁다
옆사람
퉤
쳇
의거하여
근거하여
의해
따라
힘입어
그
다음
버금
두번째로
기타
첫번째로
나머지는
그중에서
견지에서
형식으로 쓰여
입장에서
위해서
단지
의해되다
하도록시키다
뿐만아니라
반대로
전후
전자
앞의것
잠시
잠깐
하면서
그렇지만
다음에
그러한즉
그런즉
남들
아무거나
어찌하든지
같다
비슷하다
예컨대
이럴정도로
어떻게
만약
만일
위에서 서술한바와같이
인 듯하다
하지 않는다면
만약에
무엇
무슨
어느
어떤
아래윗
조차
한데
그럼에도 불구하고
여전히
심지어
까지도
조차도
하지 않도록
않기 위하여
때
시각
무렵
시간
동안
어때
어떠한
하여금
네
예
우선
누구
누가 알겠는가
아무도
줄은모른다
줄은 몰랏다
하는 김에
겸사겸사
하는바
그런 까닭에
한 이유는
그러니
그러니까
때문에
그
너희
그들
너희들
타인
것
것들
너
위하여
공동으로
동시에
하기 위하여
어찌하여
무엇때문에
붕붕
윙윙
나
우리
엉엉
휘익
윙윙
오호
아하
어쨋든
만 못하다	하기보다는
차라리
하는 편이 낫다
흐흐
놀라다
상대적으로 말하자면
마치
아니라면
쉿
그렇지 않으면
그렇지 않다면
안 그러면
아니었다면
하든지
아니면
이라면
좋아
알았어
하는것도
그만이다
어쩔수 없다
하나
일
일반적으로
일단
한켠으로는
오자마자
이렇게되면
이와같다면
전부
한마디
한항목
근거로
하기에
아울러
하지 않도록
않기 위해서
이르기까지
이 되다
로 인하여
까닭으로
이유만으로
이로 인하여
그래서
이 때문에
그러므로
그런 까닭에
알 수 있다
결론을 낼 수 있다
으로 인하여
있다
어떤것
관계가 있다
관련이 있다
연관되다
어떤것들
에 대해
이리하여
그리하여
여부
하기보다는
하느니
하면 할수록
운운
이러이러하다
하구나
하도다
다시말하면
다음으로
에 있다
에 달려 있다
우리
우리들
오히려
하기는한데
어떻게
어떻해
어찌됏어
어때
어째서
본대로
자
이
이쪽
여기
이것
이번
이렇게말하자면
이런
이러한
이와 같은
요만큼
요만한 것
얼마 안 되는 것
이만큼
이 정도의
이렇게 많은 것
이와 같다
이때
이렇구나
것과 같이
끼익
삐걱
따위
와 같은 사람들
부류의 사람들
왜냐하면
중의하나
오직
오로지
에 한하다
하기만 하면
도착하다
까지 미치다
도달하다
정도에 이르다
할 지경이다
결과에 이르다
관해서는
여러분
하고 있다
한 후
혼자
자기
자기집
자신
우에 종합한것과같이
총적으로 보면
총적으로 말하면
총적으로
대로 하다
으로서
참
그만이다
할 따름이다
쿵
탕탕
쾅쾅
둥둥
봐
봐라
아이야
아니
와아
응
아이
참나
년
월
일
령
영
일
이
삼
사
오
육
륙
칠
팔
구
이천육
이천칠
이천팔
이천구
하나
둘
셋
넷
다섯
여섯
일곱
여덟
아홉
령
영"""

In [ ]:
stopwords = stop_words.split()

In [ ]:
print(stopwords)

# 전처리 함수

In [ ]:
!pip install konlpy
# !curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh | bash -x

In [ ]:
import hanja
import re
# from konlpy.tag import Okt
# okt = Okt()
from konlpy.tag import Komoran
komoran = Komoran() 
# from konlpy.tag import Mecab
# mecab = Mecab()

In [ ]:
def preprocessing(review):
    # 1. 한자 변환
    review_text = hanja.translate(review, 'substitution')    
    # 2. http 링크 제거
    review_text = re.sub(r'http://[a-zA-Z0-9.?/&=:]*', " ", review_text)
    # 3. tag 및 특수문자 제거
    review_text = re.sub("[^가-힣ㄱ-ㅎㅏ-ㅣ0-9a-zA-Z]", " ", review_text)
    # 4. 불용어 제거
    stops = set(stopwords)
    words = [w for w in review_text.split() if not w in stops]
    review_text = ' '.join(words)    
    # 5. 형태소 분리
#     words = okt.morphs(review_text, norm=True)
    words = komoran.morphs(review_text)
#     words = mecab.morphs(review_text)
    clean_review = ' '.join(words)     
    return clean_review

In [ ]:
# 전체 데이터 전처리
train_data['clean_review']  =  train_data['document'].astype(str).apply(lambda x : preprocessing(x))

# word2vec

In [ ]:
from gensim.models import Word2Vec

In [ ]:
model = Word2Vec(sentences = [s.split() for s in train_data['clean_review']],
                              vector_size = 128, # 임의로 설정
                              window = 5, # 다섯 개 묶음으로 분석한다
                              min_count = 5,# 최소 다섯 개의 데이터를 보겠다
                              workers =4,# 동시에 4개를 
                              sg=0 # skipgram이 아니다. CBOW다
)

In [ ]:
model.wv.most_similar('영화') 

In [ ]:
model.wv.most_similar('재미') 

In [ ]:
model.wv.most_similar('별점') 

In [ ]:
model.wv['별점']

In [ ]:
train_data[['clean_review', 'label']]

In [ ]:
train_word_count = train_data['clean_review'].astype(str).apply(lambda x:len(x.split()))

In [ ]:
plt.hist(train_word_count, bins=200)

In [ ]:
train_word_count.describe()

# doc2vec

In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from nltk.tokenize import word_tokenize

In [ ]:
tagged_data = [TaggedDocument(words=word_tokenize(review), tags=[str(i)]) for i, review in enumerate(train_data['clean_review'].astype(str))] 

In [ ]:
tagged_data

In [ ]:
max_epochs=10
vec_size=2
model = Doc2Vec(vector_size=vec_size, epochs=max_epochs)

In [ ]:
model.build_vocab(tagged_data)

In [ ]:
model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)

In [ ]:
model.save("d2v.model")
model = Doc2Vec.load("d2v.model")

In [ ]:
model.infer_vector(['더빙', '진짜', '짜증', '나', '네요', '목소리'])

In [ ]:
model.infer_vector(['나름', '심오', '하', 'ㄴ', '뜻', '도', '있', '는', '듯', '그냥', '학생', '이', '선생', '과', '놀아나', '는', '영화', '는', '절대', '아니', 'ㅁ'])

In [ ]:
model.infer_vector("안녕하세요".split())

# 유사한 문서 찾기

In [ ]:
from gensim.utils import simple_preprocess

In [ ]:
vector = model.infer_vector(['나름', '심오', '하', 'ㄴ', '뜻', '도', '있', '는', '듯', '그냥', '학생', '이', '선생', '과', '놀아나', '는', '영화', '는', '절대', '아니', 'ㅁ'])

In [ ]:
return_doc = model.dv.most_similar(positive=[vector], topn=5)

In [ ]:
return_doc

In [ ]:
for doc in return_doc:
    print(train_data.iloc[int(doc[0])]['document'])

In [ ]:
model.dv.vectors

In [ ]:
train_data['doc2vec_x'] = model.dv.vectors[:,0]
train_data['doc2vec_y'] = model.dv.vectors[:,1]

In [ ]:
train_data

In [ ]:
from sklearn.cluster import KMeans
kmeans = KMeans(init='k-means++', n_clusters=2, n_init=12)

In [ ]:
kmeans.fit(model.dv.vectors)

In [ ]:
kmeans_labels = kmeans.labels_
print(f'kmeans_labels:{kmeans_labels}')

kmeans_cluster_centers = kmeans.cluster_centers_
print(f'kmeans_cluster_centers:{kmeans_cluster_centers}')

In [ ]:
train_data['cluster_label'] = kmeans_labels

In [ ]:
train_data

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df_center = pd.DataFrame(kmeans.cluster_centers_)
display(df_center)

In [ ]:
sns.lmplot( x='doc2vec_x' , y='doc2vec_y', data=train_data, fit_reg=False, hue="cluster_label" )
sns.scatterplot(x=df_center[0], y=df_center[1],  data=df_center ,s=100 )
plt.title('k-mean cluster')

In [ ]:
sns.relplot( x='doc2vec_x' , y='doc2vec_y', data=train_data, hue="label" )
plt.title('dataset label')

In [ ]:
sns.relplot( x='doc2vec_x' , y='doc2vec_y', data=train_data, col='label', hue="label" )
plt.title('dataset label')

In [ ]:
from gensim.models import Word2Vec

In [ ]:
model = Word2Vec(sentences = [s.split() for s in train_data['clean_review']],
                              vector_size = 128, # 임의로 설정
                              window = 5, # 다섯 개 묶음으로 분석한다
                              min_count = 5,# 최소 다섯 개의 데이터를 보겠다
                              workers =4,# 동시에 4개를 
                              sg=0 # skipgram이 아니다. CBOW다
)

# convolution 1D for NLP classification

In [ ]:
# data 변환용.
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# data 분할 train, validation, test
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, Dropout, Conv1D, GlobalMaxPooling1D, Dense, Input, Flatten, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import load_model

In [ ]:
X = train_data['clean_review']
y = train_data['label']

In [ ]:
w2vmodel = Word2Vec(sentences = [s.split() for s in train_data['clean_review']],
                              vector_size = 128, # 임의로 설정
                              window = 5, # 다섯 개 묶음으로 분석한다
                              min_count = 5,# 최소 다섯 개의 데이터를 보겠다
                              workers =4,# 동시에 4개를 
                              sg=0 # skipgram이 아니다. CBOW다
)

In [ ]:
vocab_size = len(w2vmodel.wv)
print(vocab_size)
max_len = 42  # 문장 최대길이 75%

In [ ]:
tokenizer = Tokenizer(vocab_size)
tokenizer.fit_on_texts(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)

In [ ]:
print(len(X))
print(len(X_train))
print(len(X_test))

In [ ]:
X_train = tokenizer.texts_to_sequences(X_train)
X_train = pad_sequences(X_train, maxlen=max_len)

X_test = tokenizer.texts_to_sequences(X_test)
X_test = pad_sequences(X_test, maxlen=max_len)

In [ ]:
embedding_dim = 128
dropout_ratio = (0.5, 0.8)
num_filters=128
hidden_units = 128

In [ ]:
model_input = Input(shape = (max_len, ))
z = Embedding(vocab_size, embedding_dim, input_length=max_len, name="embedding")(model_input)
z = Dropout(dropout_ratio[0])(z)

In [ ]:
conv_blocks = []
#kernel 3, 4, 5 layer 추가
for sz in [3,4,5]:
    conv = Conv1D(filters = num_filters,
                 kernel_size=sz,
                 padding="valid",
                 activation="relu",
                 strides=1
                 )(z)
    conv = GlobalMaxPooling1D()(conv)
    conv_blocks.append(conv)

In [ ]:
z = Concatenate()(conv_blocks) if len(conv_blocks) > 0 else conv_blocks[0]
z = Dropout(dropout_ratio[1])(z)
z = Dense(hidden_units, activation="relu")(z)
model_output = Dense(1, activation="sigmoid")(z)

In [ ]:
model = Model(model_input, model_output)
model.compile(loss="binary_crossentropy",
              optimizer="adam",
              metrics=["acc"])

es = EarlyStopping(monitor='val_loss', mode="min", verbose=1, patience=4)
mc = ModelCheckpoint("CNN_NSMC_model.h5", monotor='val_acc', mode="max", verbose=1, save_best_only=True)

In [ ]:
# model train
model.fit(X_train, y_train, batch_size=64, epochs=10, validation_split=0.2, verbose=2, callbacks=[es, mc])

In [ ]:
loaded_model = load_model("CNN_NSMC_model.h5")
print(f' \n테스트 정확도 : {loaded_model.evaluate(X_test, y_test)[1]}')

# model의 사용

In [ ]:
def sentiment_predict(new_sentence):
  new_sentence = preprocessing(new_sentence)
  encoded = tokenizer.texts_to_sequences([new_sentence]) # 정수 인코딩
  pad_new = pad_sequences(encoded, maxlen = max_len) # 패딩
  score = float(loaded_model.predict(pad_new)) # 예측
  if(score > 0.5):
    print("{:.2f}% 확률로 긍정 리뷰입니다.\n".format(score * 100))
  else:
    print("{:.2f}% 확률로 부정 리뷰입니다.\n".format((1 - score) * 100))

In [ ]:
sentiment_predict('이 영화 개꿀잼 ㅋㅋㅋ')
sentiment_predict('이딴게 영화냐 ㅉㅉ')
sentiment_predict('감독 뭐하는 놈이냐?')